# R&D 특구 유사 과제 검색 프로토타입

준비 중인 연구 주제를 입력하면, R&D 특구 소속 기관이 수행한 유사 과제와 그 과제를 수행한 연구자를
찾아줍니다.

In [1]:
import pandas as pd

DATA_DIR = "../01_data"

companies = pd.read_csv(f"{DATA_DIR}/특구입주기업현황.csv", encoding="cp949")
tasks_raw = pd.read_csv(f"{DATA_DIR}/이알앤디_과제정보.csv", encoding="cp949")

print("특구입주기업현황:", len(companies), "건")
print("이알앤디_과제정보 (원본):", len(tasks_raw), "건")

tasks = tasks_raw[tasks_raw["보안과제여부"] != "Y"].copy()
print("보안과제 제외 후:", len(tasks), "건")
assert len(tasks) == 11783, f"보안과제 제외 후 건수가 예상과 다름: {len(tasks)}"

특구입주기업현황: 6359 건
이알앤디_과제정보 (원본): 11788 건
보안과제 제외 후: 11783 건


In [2]:
companies = companies.rename(columns={"지역": "특구지역"})
companies["_key"] = companies["기관명"].str.strip()
tasks["_key"] = tasks["주관기관명"].str.strip()

companies_dedup = companies.drop_duplicates(subset="_key")

df = (
    tasks.merge(companies_dedup[["_key", "특구지역"]], on="_key", how="inner")
    .drop(columns=["_key"])
    .reset_index(drop=True)
)

print("조인 결과:", len(df), "건 /", df["주관기관명"].nunique(), "개 기관")
assert len(df) == 1680, f"조인 건수가 예상과 다름: {len(df)}"
assert df["주관기관명"].nunique() == 27, f"조인 기관 수가 예상과 다름: {df['주관기관명'].nunique()}"
assert df["연구자번호"].nunique() == 1464, f"연구자 수가 예상과 다름: {df['연구자번호'].nunique()}"

조인 결과: 1680 건 / 27 개 기관


In [3]:
_name_org_ids = (
    df.drop_duplicates(subset=["연구책임자명", "주관기관명", "연구자번호"])
    .groupby(["연구책임자명", "주관기관명"])["연구자번호"]
    .nunique()
)
DUP_NAME_ORG_PAIRS = set(_name_org_ids[_name_org_ids > 1].index)
print("동명이인 표기 대상 (이름+기관) 쌍:", len(DUP_NAME_ORG_PAIRS))
assert len(DUP_NAME_ORG_PAIRS) == 11, f"동명이인 쌍 수가 예상과 다름: {len(DUP_NAME_ORG_PAIRS)}"

def display_researcher_name(row, dup_pairs=DUP_NAME_ORG_PAIRS):
    key = (row["연구책임자명"], row["주관기관명"])
    if key in dup_pairs:
        return f"{row['연구책임자명']}({str(int(row['연구자번호']))[-4:]})"
    return row["연구책임자명"]

동명이인 표기 대상 (이름+기관) 쌍: 11


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df["과제명"])
print("TF-IDF 행렬 크기:", tfidf_matrix.shape)

svd = TruncatedSVD(n_components=2, random_state=42)
coords = svd.fit_transform(tfidf_matrix)

N_CLUSTERS = 6  # Task 9에서 화면2 산점도를 보고 최종 확정

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
clusters = kmeans.fit_predict(coords)

df["_cluster"] = clusters
print("클러스터별 건수:")
print(df["_cluster"].value_counts().sort_index())

TF-IDF 행렬 크기: (1680, 8037)


클러스터별 건수:
_cluster
0    253
1    463
2     84
3    584
4    160
5    136
Name: count, dtype: int64


In [5]:
sizes = df["_cluster"].value_counts()
assert len(sizes) == N_CLUSTERS, f"빈 클러스터 존재: {len(sizes)}/{N_CLUSTERS}"
assert (sizes > 0).all()

In [6]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

SIMILARITY_THRESHOLD = 0.15  # Task 9에서 최종 확정
TOP_N_TASKS = 5              # Task 9에서 최종 확정

def search_similar_tasks(query, threshold=SIMILARITY_THRESHOLD, top_n=TOP_N_TASKS):
    if len(df) == 0:
        return {"empty_corpus": True}

    if query is None or not query.strip():
        return {"empty_corpus": False, "blank_query": True}

    query_vec = vectorizer.transform([query])
    sims = cosine_similarity(query_vec, tfidf_matrix).flatten()
    query_coord = svd.transform(query_vec)
    query_cluster = int(kmeans.predict(query_coord)[0])

    mask = sims >= threshold
    matched_idx = np.where(mask)[0]

    if len(matched_idx) == 0:
        fallback_idx = np.argsort(-sims)[:3]
        table = df.iloc[fallback_idx][["과제명", "주관기관명", "선정년도"]].copy()
        table["유사도"] = sims[fallback_idx]
        table["클러스터"] = df.iloc[fallback_idx]["_cluster"].values
        return {
            "empty_corpus": False, "blank_query": False, "reference_only": True,
            "count": 0, "orgs": [], "table": table, "cluster": query_cluster, "sims": sims,
        }

    ordered = matched_idx[np.argsort(-sims[matched_idx])][:top_n]
    table = df.iloc[ordered][["과제명", "주관기관명", "선정년도"]].copy()
    table["유사도"] = sims[ordered]
    table["클러스터"] = df.iloc[ordered]["_cluster"].values
    return {
        "empty_corpus": False, "blank_query": False, "reference_only": False,
        "count": len(matched_idx),
        "orgs": sorted(df.iloc[matched_idx]["주관기관명"].unique().tolist()),
        "table": table, "cluster": query_cluster, "sims": sims,
    }

print("search_similar_tasks 함수 정의 완료")

search_similar_tasks 함수 정의 완료


In [7]:
print("=== 예시 3문장 테스트 ===")
for q in ["인공지능 기반 이미지 분석 기술 개발", "이차전지 소재 개발 연구", "탄소중립 에너지 저장 시스템 개발"]:
    r = search_similar_tasks(q)
    print(q, "-> count:", r["count"], "reference_only:", r["reference_only"], "cluster:", r["cluster"])
    assert r["count"] > 0 or r["reference_only"], f"'{q}'에 대해 표시할 행이 없음"

print("\n=== 엣지 케이스 테스트 ===")
for q in ["", "   ", "가", "a"]:
    r = search_similar_tasks(q)
    print(repr(q), "-> blank:", r.get("blank_query"), "reference_only:", r.get("reference_only"))

print("\n=== 모든 테스트 통과 ===")

=== 예시 3문장 테스트 ===
인공지능 기반 이미지 분석 기술 개발 -> count: 30 reference_only: False cluster: 2
이차전지 소재 개발 연구 -> count: 8 reference_only: False cluster: 2
탄소중립 에너지 저장 시스템 개발 -> count: 7 reference_only: False cluster: 2

=== 엣지 케이스 테스트 ===
'' -> blank: True reference_only: None
'   ' -> blank: True reference_only: None
'가' -> blank: False reference_only: True
'a' -> blank: False reference_only: True

=== 모든 테스트 통과 ===


In [8]:
TOP_N_RESEARCHERS = 5  # Task 9에서 최종 확정
RANK_OPTIONS = ["최고 유사도", "유사 과제 건수", "합산 점수"]

def recommend_researchers(sims, threshold=SIMILARITY_THRESHOLD, rank_by="최고 유사도", top_n=TOP_N_RESEARCHERS):
    work = df.copy()
    work["유사도"] = sims
    work["임계값이상"] = work["유사도"] >= threshold

    counts_above = work.groupby("연구자번호")["임계값이상"].sum()
    idxmax_all = work.groupby("연구자번호")["유사도"].idxmax()
    rep = work.loc[
        idxmax_all,
        ["연구자번호", "연구책임자명", "주관기관명", "특구지역", "과제명", "유사도"],
    ].rename(columns={"과제명": "대표 유사 과제명"}).set_index("연구자번호")
    sum_above = work[work["임계값이상"]].groupby("연구자번호")["유사도"].sum()

    researchers = rep.copy()
    researchers["과제수"] = counts_above.reindex(researchers.index).fillna(0).astype(int)
    researchers["합산점수"] = sum_above.reindex(researchers.index).fillna(0.0)
    researchers = researchers.reset_index()

    qualifying = researchers[researchers["과제수"] > 0]

    if len(qualifying) == 0:
        selected = researchers.sort_values("유사도", ascending=False).head(3)
        reference_only = True
    else:
        if rank_by == "유사 과제 건수":
            selected = qualifying.sort_values(["과제수", "유사도"], ascending=[False, False])
        elif rank_by == "합산 점수":
            selected = qualifying.sort_values("합산점수", ascending=False)
        else:
            selected = qualifying.sort_values("유사도", ascending=False)
        selected = selected.head(top_n)
        reference_only = False

    selected = selected.copy()
    selected["연구책임자명"] = selected.apply(display_researcher_name, axis=1)
    display_cols = ["연구책임자명", "주관기관명", "특구지역", "대표 유사 과제명", "유사도", "과제수"]
    return {"reference_only": reference_only, "table": selected[display_cols]}

print("recommend_researchers 함수 정의 완료")

recommend_researchers 함수 정의 완료


In [9]:
print("=== 예시 3문장 × 랭킹 3종 = 9회 호출 확인 ===")
example_queries = ["인공지능 기반 이미지 분석 기술 개발", "이차전지 소재 개발 연구", "탄소중립 에너지 저장 시스템 개발"]

for q in example_queries:
    r_search = search_similar_tasks(q)
    for rank_by in RANK_OPTIONS:
        r_rec = recommend_researchers(r_search["sims"], rank_by=rank_by)
        n_rows = len(r_rec["table"])
        print(q, "|", rank_by, "-> rows:", n_rows, "reference_only:", r_rec["reference_only"])
        assert n_rows >= 1, f"'{q}' / '{rank_by}' 조합에서 연구자 추천 행이 0개"

print("\n=== 모든 9회 조합 테스트 통과 ===")

=== 예시 3문장 × 랭킹 3종 = 9회 호출 확인 ===
인공지능 기반 이미지 분석 기술 개발 | 최고 유사도 -> rows: 5 reference_only: False
인공지능 기반 이미지 분석 기술 개발 | 유사 과제 건수 -> rows: 5 reference_only: False
인공지능 기반 이미지 분석 기술 개발 | 합산 점수 -> rows: 5 reference_only: False
이차전지 소재 개발 연구 | 최고 유사도 -> rows: 5 reference_only: False
이차전지 소재 개발 연구 | 유사 과제 건수 -> rows: 5 reference_only: False
이차전지 소재 개발 연구 | 합산 점수 -> rows: 5 reference_only: False
탄소중립 에너지 저장 시스템 개발 | 최고 유사도 -> rows: 5 reference_only: False
탄소중립 에너지 저장 시스템 개발 | 유사 과제 건수 -> rows: 5 reference_only: False
탄소중립 에너지 저장 시스템 개발 | 합산 점수 -> rows: 5 reference_only: False

=== 모든 9회 조합 테스트 통과 ===


In [10]:
print("=== 동명이인 표기 확인 (Task 2) ===")
researchers = df.drop_duplicates(subset=["연구자번호"])[["연구자번호", "연구책임자명", "주관기관명"]]
labeled = [r for _, r in researchers.iterrows() if "(" in display_researcher_name(r)]
print("괄호 표기가 붙은 연구자 수 (전체 1,464명 중):", len(labeled))
assert len(labeled) > 0, "동명이인 표기 로직이 한 번도 발생하지 않음 — 로직 점검 필요"
print("=== 동명이인 표기 확인 통과 ===")

=== 동명이인 표기 확인 (Task 2) ===
괄호 표기가 붙은 연구자 수 (전체 1,464명 중): 22
=== 동명이인 표기 확인 통과 ===


In [11]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

text_input = widgets.Text(description="과제 주제:", placeholder="준비 중인 과제의 주제를 입력하세요", layout=widgets.Layout(width="500px"))
rank_dropdown = widgets.Dropdown(options=RANK_OPTIONS, value="최고 유사도", description="랭킹 기준:")
search_button = widgets.Button(description="검색", button_style="primary")
output_area = widgets.Output()

def render_results(query, rank_by):
    with output_area:
        clear_output()
        result = search_similar_tasks(query)

        if result.get("empty_corpus"):
            print("특구 소속 기관이 수행한 과제 데이터가 없습니다.")
            return
        if result.get("blank_query"):
            print("검색어를 입력해주세요.")
            return

        print(f"내 주제가 속한 것으로 추정되는 클러스터 번호: {result['cluster']}")

        if result["reference_only"]:
            print("정확히 일치하는 유사 과제를 찾지 못했습니다. 아래는 유사도가 가장 높았던 3건입니다(참고용).")
        else:
            print(f"유사 과제 건수: {result['count']}건")
            print("수행 기관:", ", ".join(result["orgs"]))
        display(result["table"])

        rec = recommend_researchers(result["sims"], rank_by=rank_by)
        if rec["reference_only"]:
            print("참고용 — 임계값 미달: 유사도가 가장 높은 연구자 3명입니다.")
        print("연구자 추천")
        display(rec["table"])

def on_search_clicked(_):
    render_results(text_input.value, rank_dropdown.value)

def on_rank_changed(_):
    render_results(text_input.value, rank_dropdown.value)

search_button.on_click(on_search_clicked)
rank_dropdown.observe(on_rank_changed, names="value")

display(widgets.HBox([text_input, rank_dropdown, search_button]))
display(output_area)

Output()

In [12]:
for q in ["인공지능 기반 이미지 분석 기술 개발", "이차전지 소재 개발 연구", "탄소중립 에너지 저장 시스템 개발", "", "가"]:
    text_input.value = q
    for rank_by in RANK_OPTIONS:
        rank_dropdown.value = rank_by
        on_search_clicked(None)  # output_area에 렌더링, 예외 발생 시 여기서 즉시 실패
print("모든 조합 실행 완료 — 예외 없음")

모든 조합 실행 완료 — 예외 없음
